# Split Point-Cloud Viewer (diagnostic tool)

Compare **two point clouds side by side** in a desktop (tkinter) window: five spatial
views each, per-field statistics, colouring by any data field, and value/range
filtering. Built to answer questions like *"which fields should be merged?"* and
*"how do the class labels of dataset A map onto dataset B?"* — the questions the
`class_unifier` stage exists to settle.

**Per-panel interactions**

- **Fields shown…** — pick which fields the statistics box lists (all shown by
  default; untick the constant/uninformative ones to cut the clutter).
- **Colours…** — set the colour of **every value** of the current **Colour by**
  field: a palette dropdown, a hex box, the OS colour picker, and Save/Load
  (see *Colours* below).
- **3D view** — open the **currently filtered selection** in a separate orbitable
  window (see below). 2-D stays the default view.
- **Histogram** / **Unique…** — pop up a histogram, or a sortable *value / count /
  %* table, of the current **Colour by** field. Each pop-up has a *respect active
  filter* toggle (off = whole cloud). Use **Unique…** to discover which of a wide
  id field (e.g. `tree_ID`, 0…~60 000) actually exist before filtering to one.
- **Mouse wheel** — zoom the spatial view under the cursor; the matplotlib toolbar
  Home button resets it.
- For `.las`/`.laz`, the info box header line also shows the **LAS version** and
  point-format id, e.g. `LAS/LAZ  1.4 (PF 6)`.

**The 3-D view**

`misc/view_cloud_3d.py` renders the selection with **PyVista + VTK** in its own
window: left-drag rotates, scroll zooms, middle-drag pans, `r` resets the camera,
`d` toggles depth shading, `q` closes.

- It runs as a **separate process**, so the 2-D window stays responsive and Cloud A
  and Cloud B can each have a 3-D window open at the same time.
- It shows the points the filter selected, capped by its own **3D max pts** budget
  (default 500 000 — the GPU takes far more than the 2-D scatter).
- Colours match the 2-D views exactly (same `color_spec` rule and the same colour
  scheme), so anything you set in **Colours…** shows up here too: a discrete class
  legend for ≤ 20 integer labels, otherwise a viridis scalar bar.
- Requires `pyvista` + `vtk`. If they are missing the button explains how to install
  them. *(Open3D is not used: it publishes no wheels for this env's Python 3.13.)*

**⚠️ Integer columns colour as classes, float columns do not**

The rule behind every colour decision here is
`discrete = np.issubdtype(dtype, np.integer) and n_unique <= 20`, applied to **what is
currently drawn**. A label column that arrives as **float** can never satisfy it, so it
is drawn as a continuous viridis ramp with **no class legend**, its *Unique* cell shows
`-`, and its histogram falls back to generic bins.

That is why `class_unifier` defaults to **`output_format: ply`**: a PLY stores a dtype
per property, so `semantic_seg` stays `uint8` and `tree_ID` stays an int. Its `.npy`
option is a single float64 matrix and views poorly for exactly this reason — load the
`.ply` when you want to *see* classes.

Because the rule looks at what is drawn, it also works the other way round: filter a
wide id field such as `tree_ID` down to a handful of ids and those few get their own
discrete colours and a legend.

**Colours — one colour per value**

- **A value's colour is pinned the first time it is drawn**, and then nothing moves it
  except you. Filtering, the random downsample and the other panel all leave it alone.
  (It used to be handed out by a value's *rank among the values on screen*, so
  filtering to `{3,4}` silently recoloured every class — measured on class 3: light
  orange over `{0..4}`, blue after that filter, light blue in a cloud without classes
  0 and 2.)
- **Both panels share one colour table**, keyed by field name, so class 3 is the same
  colour in Cloud A and Cloud B even when the two clouds contain different values —
  which is the whole point of comparing them. Changing a colour redraws both.
- **Hex or name.** `#E9E56B`, `#abc`, `e9e56b` (no `#`) and matplotlib names like `red`
  or `tab:blue` all work. A bare number is rejected on purpose — matplotlib would read
  `1` as white and `0.5` as grey, which is never what a hex box means. Bad input turns
  the box red and keeps the old colour.
- **Palettes.** `tab20` (the default), `tab10`, `Set1`, `Set2`, `Set3`, `Paired`,
  `Dark2`, `Accent`, `Pastel1`, plus **ForAINet classes** — the colours ForAINet itself
  uses for its output PLYs, matching our on-disk `semantic_seg`: 0 unclassified black,
  1 low_vegetation yellow, 2 ground blue, 3 stem_points brown, 4 live_branches salmon.
  *Apply palette* refills every known value of the field.
- **Save… / Load…** write one small JSON per field, by default into
  `misc/conf/colours/<field>.json`:

  ```json
  {"field": "semantic_seg", "palette": "ForAINet classes",
   "colours": {"0": "#000000", "1": "#e9e56b", "2": "#5f9cc4"}}
  ```

  Loading merges the file in and reports any unusable entry rather than skipping it
  silently. The file is plain enough to hand-edit.
- Per-value colours need a field that is currently drawn as classes. On a float or
  wide field the button explains how to get there (filter it down) instead of doing
  nothing.

**Supported formats**

| Format | How fields are found |
|---|---|
| `.las` / `.laz` | all point dimensions (incl. extra dims like `tree_ID`, `dist_axes`, `Class`); `x/y/z` are the scaled coordinates; header line shows version + point format |
| `.ply` | all vertex properties, each with its own dtype — **the best format to inspect** |
| `.npy` (structured array) | names taken from the array's dtype |
| `.npy` (plain N×K matrix) | names taken from a **JSON sidecar** (see below); every column is float64 |

**The `.npy` JSON sidecar convention**

A bare `.npy` matrix stores no column names, so the viewer looks for a JSON file with
the **same stem** next to it — `plot_01_007.npy` → `plot_01_007.json`:

```json
{"columns": ["x", "y", "z", "2tree_ID"]}
```

Any *additional* keys in that JSON are treated as metadata about the cloud and shown
in the info box. If the sidecar is missing, unreadable, or its `columns` count does
not match the matrix, the viewer falls back to `x, y, z, col_3, …` and shows a
warning. `class_unifier` writes these sidecars (alongside either output format), so
its exports carry their column names and full provenance.

**How to run**

1. Use the **aifor** kernel (`mamba env` with `laspy`, `plyfile`, `matplotlib`, and
   `pyvista`+`vtk` for the 3-D view).
2. Run the cells top to bottom; the last cell opens the viewer window.
3. ⚠️ The notebook kernel is **busy while the window is open** (tkinter's event loop
   runs in the kernel). Close the window to get the kernel back. The 3-D windows are
   separate processes and are *not* affected by this.

In [16]:
"""Imports.

matplotlib is embedded directly into the tkinter window via ``FigureCanvasTkAgg``:
we build ``Figure`` objects ourselves and never touch ``pyplot`` / ``plt.show()``.
Keeping pyplot (and its hidden global state) out of the picture is the recommended
way to host matplotlib inside a GUI toolkit.

``matplotlib.colors`` (``to_hex`` / ``to_rgb`` / ``to_rgba``) is what parses the colours
typed into the **Colours…** editor, so ``#E9E56B``, ``#abc`` and names like ``red`` all
work without hand-written hex parsing. ``tkinter.colorchooser`` is the OS colour picker
behind its *Pick…* buttons.

PyVista/VTK (the 3-D view) is deliberately NOT imported here: it runs in a separate
process (``misc/view_cloud_3d.py``) because VTK and tkinter each want to own the
event loop. See ``CloudPanel.open_3d_view``.
"""
import importlib.util
import json
import logging
import re
import subprocess
import sys
import tempfile
import tkinter as tk
from pathlib import Path
from tkinter import colorchooser, filedialog, messagebox, ttk
from tkinter.scrolledtext import ScrolledText

import laspy
import numpy as np
import pandas as pd
from matplotlib import colormaps
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.colors import to_hex, to_rgb, to_rgba
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from plyfile import PlyData

print("imports OK")

imports OK


In [17]:
# ---------------------------------------------------------------------------
# Loaders: every supported format is normalised into ONE structure so the GUI
# (and any code you write in later cells) never has to care where a cloud
# came from:
#
#   {
#     "name":       "plot_01_tree_ID_dist_axes.las",   # file name (display)
#     "path":       Path(...),                          # full path
#     "file_type":  "LAS/LAZ" | "PLY" | "NPY",
#     "num_points": 18_123_456,
#     "fields":     {"x": ndarray, "y": ndarray, ...},  # plain 1-D numpy arrays
#     "meta":       {...},   # extra sidecar keys (NPY only), shown in info box
#     "header":     {...},   # format-specific header facts (LAS version, ...)
#     "warnings":   [...],   # human-readable problems, shown in info box
#   }
#
# laspy gotcha (repo convention): laspy exposes x/y/z as ScaledArrayView and
# bit fields (return_number, ...) as SubFieldView. Those are NOT plain arrays
# and confuse numpy/pandas, so every accessor is materialised with np.asarray.
# ---------------------------------------------------------------------------

# Raw integer coordinate dimensions of the LAS spec. We drop them and expose the
# scaled x/y/z instead (raw ints are scale*X+offset away from real coordinates).
_LAS_RAW_COORDS = ("X", "Y", "Z")


def _load_las(path: Path) -> dict:
    """Read a .las/.laz file into the common cloud structure."""
    las = laspy.read(str(path))
    fields = {
        "x": np.asarray(las.x, dtype=np.float64),
        "y": np.asarray(las.y, dtype=np.float64),
        "z": np.asarray(las.z, dtype=np.float64),
    }
    # Every remaining dimension, including extra-bytes dims (tree_ID, dist_axes,
    # Class, ...) that 3DFin / forainet_prep attach.
    for dim in las.point_format.dimension_names:
        if dim in _LAS_RAW_COORDS:
            continue
        fields[dim] = np.asarray(getattr(las, dim))
    return {
        "name": path.name,
        "path": path,
        "file_type": "LAS/LAZ",
        "num_points": len(fields["x"]),
        "fields": fields,
        "meta": {},
        # Header facts worth surfacing in the info box. Same idiom as
        # pipeline/forainet_prep.py: version as a string ("1.4"), point format id.
        "header": {
            "las_version": str(las.header.version),
            "point_format": int(las.point_format.id),
        },
        "warnings": [],
    }


def _load_ply(path: Path) -> dict:
    """Read a .ply file (all vertex properties) into the common cloud structure."""
    ply = PlyData.read(str(path))
    vertex = ply["vertex"].data
    fields = {name: np.asarray(vertex[name]) for name in vertex.dtype.names}
    return {
        "name": path.name,
        "path": path,
        "file_type": "PLY",
        "num_points": len(vertex),
        "fields": fields,
        "meta": {},
        "header": {},
        "warnings": [],
    }


def _npy_column_names(path: Path, n_cols: int) -> tuple:
    """Resolve column names for a bare N x K .npy matrix.

    Looks for a JSON sidecar with the same stem (plot_01_007.npy ->
    plot_01_007.json) holding at least {"columns": [...]}. Every OTHER key of
    that JSON is returned as metadata so the GUI can display it. Falls back to
    generic names (x, y, z, col_3, ...) whenever the sidecar is unusable, and
    reports what happened through the warnings list.

    Returns (names, meta, warnings).
    """
    warnings, meta = [], {}
    names = None

    sidecar = path.with_suffix(".json")
    if sidecar.exists():
        try:
            info = json.loads(sidecar.read_text(encoding="utf-8"))
            cols = info.get("columns")
            if isinstance(cols, list) and len(cols) == n_cols:
                names = [str(c) for c in cols]
                meta = {k: v for k, v in info.items() if k != "columns"}
            else:
                got = len(cols) if isinstance(cols, list) else repr(cols)
                warnings.append(
                    f"sidecar {sidecar.name}: 'columns' does not match the array "
                    f"({got} names vs {n_cols} columns)"
                )
        except (OSError, json.JSONDecodeError) as exc:
            warnings.append(f"sidecar {sidecar.name} could not be read: {exc}")
    else:
        warnings.append(f"no sidecar {sidecar.name} next to the array")

    if names is None:
        names = ["x", "y", "z"][:n_cols] + [f"col_{i}" for i in range(3, n_cols)]
        warnings.append("using fallback column names: " + ", ".join(names))
    return names, meta, warnings


def _load_npy(path: Path) -> dict:
    """Read a .npy file (structured array or bare N x K matrix)."""
    arr = np.load(str(path), allow_pickle=False)
    meta, warnings = {}, []

    if arr.dtype.names:
        # Structured array: numpy already stored the column names for us.
        fields = {name: np.asarray(arr[name]) for name in arr.dtype.names}
    else:
        if arr.ndim == 1:  # a single column still gets the full treatment
            arr = arr.reshape(-1, 1)
        if arr.ndim != 2:
            raise ValueError(f"{path.name}: expected an N x K matrix, got shape {arr.shape}")
        names, meta, warnings = _npy_column_names(path, arr.shape[1])
        # ascontiguousarray: column slices of a 2-D array are strided views;
        # copying them keeps all downstream numpy ops fast and predictable.
        fields = {name: np.ascontiguousarray(arr[:, i]) for i, name in enumerate(names)}

    n = len(next(iter(fields.values())))
    return {
        "name": path.name,
        "path": path,
        "file_type": "NPY",
        "num_points": n,
        "fields": fields,
        "meta": meta,
        "header": {},
        "warnings": warnings,
    }


_LOADERS = {".las": _load_las, ".laz": _load_las, ".ply": _load_ply, ".npy": _load_npy}


def load_point_cloud(path) -> dict:
    """Load any supported point-cloud file into the common structure."""
    path = Path(path)
    loader = _LOADERS.get(path.suffix.lower())
    if loader is None:
        raise ValueError(
            f"unsupported file type '{path.suffix}' "
            f"(supported: {', '.join(sorted(_LOADERS))})"
        )
    return loader(path)


def summarize_fields(cloud: dict, fields=None) -> pd.DataFrame:
    """Per-field statistics table: dtype, Min, Max, Unique.

    ``fields`` optionally restricts (and orders) the rows to a list of field
    names; ``None`` (default) reports every field in insertion order. Unknown
    names are skipped so a stale selection never raises.

    Unique counts are only computed for integer fields (labels, ids): they are
    the interesting ones for label diagnosis, and np.unique on huge float
    columns would be slow for no benefit.
    """
    names = list(cloud["fields"]) if fields is None else fields
    rows = []
    for name in names:
        col = cloud["fields"].get(name)
        if col is None:
            continue
        row = {"Field": name, "dtype": str(col.dtype), "Min": "-", "Max": "-", "Unique": "-"}
        if np.issubdtype(col.dtype, np.number):
            row["Min"] = f"{col.min():g}"
            row["Max"] = f"{col.max():g}"
            if np.issubdtype(col.dtype, np.integer):
                row["Unique"] = f"{np.unique(col).size:,}"
        rows.append(row)
    return pd.DataFrame(rows)


def cloud_info_text(cloud: dict, fields=None) -> str:
    """The full plain-text report shown in a panel's info box.

    ``fields`` restricts which per-field stat rows are shown (see
    ``summarize_fields``); the header line, sidecar metadata and warnings are
    always shown. For LAS/LAZ the header line also carries the file version and
    point-format id, e.g. ``LAS/LAZ  1.4 (PF 6)``.
    """
    header = cloud["file_type"]
    hdr = cloud.get("header") or {}
    if hdr.get("las_version"):
        header += f"  {hdr['las_version']}"
        if hdr.get("point_format") is not None:
            header += f" (PF {hdr['point_format']})"
    lines = [
        f"{header}  |  {cloud['num_points']:,} points",
        str(cloud["path"]),
        "",
        summarize_fields(cloud, fields).to_string(index=False),
    ]
    if cloud["meta"]:
        lines += ["", "Sidecar metadata:"]
        lines += [f"  {k}: {v}" for k, v in cloud["meta"].items()]
    if cloud["warnings"]:
        lines += ["", "WARNINGS:"]
        lines += [f"  ! {w}" for w in cloud["warnings"]]
    return "\n".join(lines)


print("loaders defined")

loaders defined


In [18]:
# ---------------------------------------------------------------------------
# Filtering + projection + colour helpers.
#
# These are pure functions (no GUI, no globals) on purpose: the tkinter class
# below only wires widgets to them, and they can be reused / unit-tested from
# plain notebook cells. ColourScheme is the one exception that holds state --
# it is the shared colour table, and it is still tk-free and testable.
# ---------------------------------------------------------------------------


def parse_values(text: str) -> list:
    """Parse a comma-separated values string ("2, 3, 4.5") into numbers.

    Each item is tried as int first, then float, else kept as a string (useful
    should a field ever hold non-numeric data). Empty items are skipped.
    """
    values = []
    for item in text.split(","):
        item = item.strip()
        if not item:
            continue
        try:
            values.append(int(item))
        except ValueError:
            try:
                values.append(float(item))
            except ValueError:
                values.append(item)
    return values


def make_mask(fields: dict, field: str, mode: str,
              vmin=None, vmax=None, values=None) -> np.ndarray:
    """Boolean keep-mask over all points of a cloud.

    mode="range":  keep points with vmin <= field <= vmax
                   (either bound may be None = unbounded on that side).
    mode="values": keep points whose field is in `values` (exact match),
                   e.g. Class in {2, 3}.
    """
    col = fields[field]
    if mode == "range":
        mask = np.ones(len(col), dtype=bool)
        if vmin is not None:
            mask &= col >= vmin
        if vmax is not None:
            mask &= col <= vmax
        return mask
    if mode == "values":
        if not values:
            return np.ones(len(col), dtype=bool)
        return np.isin(col, values)
    raise ValueError(f"unknown filter mode: {mode!r}")


# --------------------------------------------------------------------- colours
# Everything below decides WHICH COLOUR A VALUE GETS. One scheme object per
# window is shared by both panels and by the 3-D export, so a class cannot look
# different depending on where you look at it.

_HEX_BODY = re.compile(r"[0-9a-fA-F]{3}|[0-9a-fA-F]{6}")

# Palette names offered by the Colours... editor. tab20 is first and is the
# default, so a freshly opened cloud is coloured exactly as it always was.
# "ForAINet classes" is not a matplotlib colormap but the preset below.
FORAINET_PALETTE = "ForAINet classes"
DEFAULT_PALETTE = "tab20"
PALETTES = ("tab20", "tab10", "Set1", "Set2", "Set3", "Paired", "Dark2",
            "Accent", "Pastel1", FORAINET_PALETTE)

# ForAINet's own class colours, so the viewer can show a cloud the way ForAINet
# draws its own output PLYs. Taken from OBJECT_COLOR in
# ForAINet/PointCloudSegmentation/torch_points3d/datasets/panoptic/treeins_set1.py
# and re-indexed to the values actually stored on disk: our semantic_seg is
# 0 = unclassified (ForAINet's last "unlabelled" row, black) and 1..4 the four
# real classes (OBJECT_COLOR rows 0..3).
FORAINET_CLASS_COLOURS = {
    0: "#000000",   # unclassified  (ForAINet ignores these in the loss)
    1: "#e9e56b",   # low_vegetation - yellow
    2: "#5f9cc4",   # ground         - blue
    3: "#b37451",   # stem_points    - brown
    4: "#f19583",   # live_branches  - salmon
}


def parse_colour(text):
    """Normalise user input into ``"#rrggbb"``, or return None if it is not a colour.

    Accepts ``#RRGGBB``, ``#RGB``, either of those without the ``#``, and any
    matplotlib colour name (``red``, ``tab:blue``) -- matplotlib does the parsing,
    so there is no hand-rolled hex logic to get wrong.

    Bare numbers are REJECTED on purpose: matplotlib reads ``"1"`` as a grey level
    (white) and ``"0.5"`` as mid grey, which is never what someone typing into a
    hex box meant.
    """
    if text is None:
        return None
    s = str(text).strip()
    if not s:
        return None
    try:
        float(s)
    except ValueError:
        pass
    else:
        return None                       # "1", "0.5" -> matplotlib greys, not hex
    if _HEX_BODY.fullmatch(s):
        s = "#" + s                       # allow "e9e56b" as well as "#e9e56b"
    try:
        return to_hex(to_rgb(s))          # lowercase #rrggbb; any alpha dropped
    except ValueError:
        return None


def palette_colour(palette: str, slot: int, value=None) -> str:
    """The hex for one slot of ``palette`` (preset palettes look the value up instead).

    ``slot`` wraps around the palette, so more values than colours repeats rather
    than failing. For the ForAINet preset a value outside 0..4 has no defined
    colour and falls back to the default palette.
    """
    if palette == FORAINET_PALETTE:
        if value is not None and int(value) in FORAINET_CLASS_COLOURS:
            return FORAINET_CLASS_COLOURS[int(value)]
        palette = DEFAULT_PALETTE
    cmap = colormaps[palette]
    return to_hex(cmap(int(slot) % cmap.N))


class ColourScheme:
    """Which colour each value of each field gets. ONE instance per window.

    Both panels share it, so class 3 is the same colour in Cloud A and Cloud B --
    the whole point of a side-by-side comparison.

    **A value's colour is pinned the first time it is drawn** and then never moves
    on its own. That is deliberate. Colours used to be handed out by a value's
    *rank among the values currently on screen*, so the same class changed colour
    whenever another value appeared or disappeared; measured on value 3, it came
    out light orange over {0,1,2,3,4}, blue after filtering to {3,4}, and light
    blue in a cloud without classes 0 and 2.
    """

    def __init__(self, palette: str = DEFAULT_PALETTE):
        self.default_palette = palette
        self.palettes = {}      # field -> palette name
        self.colours = {}       # field -> {int value: "#rrggbb"}

    def palette_of(self, field: str) -> str:
        return self.palettes.get(field, self.default_palette)

    def known(self, field: str) -> dict:
        """A copy of the pinned {value: hex} for one field."""
        return dict(self.colours.get(field, {}))

    def colours_for(self, field: str, values) -> dict:
        """{value: hex} for ``values``, pinning any value not seen before.

        New values take the next free palette slot in value order, so the result
        is the same however the points arrive.
        """
        pinned = self.colours.setdefault(field, {})
        palette = self.palette_of(field)
        for value in sorted(int(v) for v in values):
            if value not in pinned:
                pinned[value] = palette_colour(palette, len(pinned), value)
        return {int(v): pinned[int(v)] for v in values}

    def set_colour(self, field: str, value, colour: str) -> str:
        """Pin one value to one colour. Raises ValueError on unparseable input."""
        hexed = parse_colour(colour)
        if hexed is None:
            raise ValueError(f"not a colour: {colour!r}")
        self.colours.setdefault(field, {})[int(value)] = hexed
        return hexed

    def apply_palette(self, field: str, palette: str, values=()) -> dict:
        """Re-fill every known value of ``field`` from ``palette``.

        "Known" is what is pinned plus ``values`` (normally the ones on screen), so
        the palette spreads over the whole field rather than just the visible part.
        """
        if palette not in PALETTES:
            raise ValueError(f"unknown palette: {palette!r}")
        self.palettes[field] = palette
        known = set(self.colours.get(field, {})) | {int(v) for v in values}
        self.colours[field] = {v: palette_colour(palette, i, v)
                               for i, v in enumerate(sorted(known))}
        return dict(self.colours[field])

    # -- saving / loading ---------------------------------------------------
    # The file is per field and deliberately boring:
    #   {"field": "semantic_seg", "palette": "ForAINet classes",
    #    "colours": {"0": "#000000", "1": "#e9e56b", ...}}
    # JSON object keys are always strings, hence the int() on the way back in.

    def to_json(self, field: str) -> dict:
        return {
            "field": field,
            "palette": self.palette_of(field),
            "colours": {str(v): c for v, c in sorted(self.known(field).items())},
        }

    def load_json(self, obj) -> tuple:
        """Merge a saved scheme in. Returns (field, n_loaded, problems).

        Every entry is validated and bad ones are REPORTED rather than dropped in
        silence -- a typo in a hand-edited file should be visible, not just absent.
        """
        if not isinstance(obj, dict):
            raise ValueError("a colour scheme must be a JSON object")
        field = obj.get("field")
        if not field:
            raise ValueError("this colour scheme has no 'field' key")

        problems = []
        palette = obj.get("palette")
        if palette in PALETTES:
            self.palettes[field] = palette
        elif palette is not None:
            problems.append(f"unknown palette {palette!r}; keeping {self.palette_of(field)!r}")

        loaded = {}
        for key, colour in (obj.get("colours") or {}).items():
            try:
                value = int(key)
            except (TypeError, ValueError):
                problems.append(f"value {key!r} is not an integer")
                continue
            hexed = parse_colour(colour)
            if hexed is None:
                problems.append(f"value {key}: not a colour: {colour!r}")
                continue
            loaded[value] = hexed
        self.colours.setdefault(field, {}).update(loaded)
        return field, len(loaded), problems


def color_spec(cvals: np.ndarray, max_classes: int = 20, scheme=None, field=None) -> dict:
    """Decide how a value column should be coloured.

    ONE rule shared by the 2-D scatter and the 3-D view, so the two can never
    disagree about what a colour means:

    * few distinct integers (<= max_classes) -> **discrete** colours plus a
      legend, the right thing for class labels and small id sets;
    * anything else -> a **continuous** viridis ramp with a colourbar.

    Note the decision is made on the values PASSED IN, i.e. on what is actually
    drawn. That is what makes "filter tree_ID down to a few ids and each tree gets
    its own colour" work.

    ``scheme`` + ``field``: a ColourScheme supplying the colour of each value. It
    has a deliberate SIDE EFFECT -- values the scheme has not seen are pinned to it
    here, which is exactly what keeps a value's colour fixed across filters,
    redraws and both panels. Without a scheme the colours fall back to tab20 by
    rank, the original behaviour, so older cells calling ``color_spec(cvals)``
    still work (their colours still shift with the value set).

    Returns a dict with:
        discrete : bool
        uniq     : the distinct values (discrete only, else None)
        palette  : (K, 4) RGBA floats, one per uniq value (discrete only)
        rgba     : (N, 4) RGBA floats, one per point (discrete only)
        rgb      : (N, 3) uint8, the same colours 0-255 (discrete only) --
                   what VTK/PyVista wants
        scalars  : (N,) float64 raw values (continuous only, else None)
    """
    cvals = np.asarray(cvals)
    discrete = (np.issubdtype(cvals.dtype, np.integer)
                and np.unique(cvals).size <= max_classes)
    if not discrete:
        return {"discrete": False, "uniq": None, "palette": None,
                "rgba": None, "rgb": None, "scalars": cvals.astype(np.float64)}

    uniq = np.unique(cvals)
    if scheme is not None and field:
        hexes = scheme.colours_for(field, uniq)
        palette = np.array([to_rgba(hexes[int(v)]) for v in uniq])
    else:
        tab20 = colormaps[DEFAULT_PALETTE]
        palette = np.array([tab20(i % tab20.N) for i in range(uniq.size)])
    rgba = palette[np.searchsorted(uniq, cvals)]
    return {
        "discrete": True,
        "uniq": uniq,
        "palette": palette,
        "rgba": rgba,
        "rgb": (rgba[:, :3] * 255).round().astype(np.uint8),
        "scalars": None,
    }


# The five spatial views. The isometric ones rotate the cloud about the z axis
# by the given angle and then look at it from the front (rotated-x vs z), i.e.
# a "camera walking around the plot" at 45 deg steps between the axis-aligned
# front/side views.
VIEWS = (
    ("XY (top)", "xy"),
    ("XZ (front)", "xz"),
    ("YZ (side)", "yz"),
    ("ISO 45\N{DEGREE SIGN}", "iso45"),
    ("ISO 135\N{DEGREE SIGN}", "iso135"),
)


def project(x: np.ndarray, y: np.ndarray, z: np.ndarray, view: str):
    """Project 3-D points onto the 2-D plane of the requested view.

    Returns (u, v): the horizontal and vertical plot coordinates.
    """
    if view == "xy":
        return x, y
    if view == "xz":
        return x, z
    if view == "yz":
        return y, z
    if view in ("iso45", "iso135"):
        theta = np.deg2rad(45.0 if view == "iso45" else 135.0)
        return x * np.cos(theta) + y * np.sin(theta), z
    raise ValueError(f"unknown view: {view!r}")


print("helpers defined")

helpers defined


In [19]:
# ---------------------------------------------------------------------------
# The tkinter GUI.
#
# Layout: one window, two identical panels (Cloud A | Cloud B). Each panel is
# fully independent -- its own file, its own colour field, its own filter --
# because the whole point is comparing two datasets whose fields DIFFER. The
# one thing they SHARE is the ColourScheme, so a value looks the same on both
# sides (see PointCloudCompareApp).
#
#   +------------------------- Cloud A -------------------------+  (same for B)
#   | [Load...]  plot_01_tree_ID_dist_axes.las                  |
#   | info box: type/version, #points, per-field stats          |
#   | [Fields shown...] [Colours...] [3D view] 3D max pts [...] |
#   | Colour by: [Class v] Max pts:[100000] [Redraw]           |
#   |            [Histogram] [Unique...]                        |
#   | Filter: [tree_ID v] (o) Range ( ) Values                  |
#   |         min [   ] max [   ] values [2, 3]  [Apply][Reset] |
#   | shown 100,000 / filtered 3,412,111 / total 18,000,000     |
#   |  [XY (top)] [XZ (front)] [YZ (side)]                      |
#   |  [ISO 45]   [ISO 135]    [legend / colourbar]             |
#   |  (mouse-wheel zooms the view under the cursor)            |
#   |  (matplotlib toolbar: zoom / pan / save PNG)              |
#   +------------------------------------------------------------+
#
# Histogram / Unique... open a pop-up for the current "Colour by" field, each
# with a "respect active filter" toggle (off = whole cloud). "Fields shown..."
# chooses which fields the statistics box lists (all shown by default).
# "Colours..." sets the colour of every value of the current "Colour by" field
# (palette, hex, OS colour picker, save/load) -- see open_colour_editor.
# "3D view" opens the CURRENT SELECTION in a separate orbitable window -- 2-D
# stays the default view, 3-D is on demand (see open_3d_view).
# ---------------------------------------------------------------------------

# Fields tried (in order) as the default colour/filter field after loading --
# the label-ish fields are what this tool is usually used to inspect.
_PREFERRED_FIELDS = ("Class", "semantic_seg", "classification",
                     "tree_ID", "treeID", "2tree_ID")
_MAX_LEGEND_CLASSES = 20     # <= this many unique ints -> discrete colours + legend
_DEFAULT_MAX_POINTS = 100_000
_DOWNSAMPLE_SEED = 42        # fixed seed -> the same subsample on every redraw
_FILE_TYPES = [("Point clouds", "*.las *.laz *.ply *.npy"), ("All files", "*.*")]
_JSON_FILE_TYPES = [("JSON", "*.json"), ("All files", "*.*")]
_ZOOM_STEP = 1.2             # mouse-wheel zoom factor per notch
_HIST_MAX_BARS = 50          # <= this many unique ints -> one histogram bar per value
_UNIQUE_ROW_CAP = 100_000    # refuse to populate a unique-values table larger than this
# The GPU handles far more points than the matplotlib scatter, so the 3-D view
# gets its own (larger) budget instead of reusing "Max points".
_DEFAULT_3D_MAX_POINTS = 500_000
_POINT_SIZE_3D = 2.0
_BAD_INPUT_FG = "#b00020"    # entry text colour when a typed value cannot be parsed


def _find_3d_script():
    """Locate misc/view_cloud_3d.py (notebooks have no __file__ to key off).

    Tries the plausible working directories: the notebook's own folder, and a
    repo root with the notebook one level down. Returns None when not found, so
    the button can explain the problem instead of raising.
    """
    for candidate in (Path.cwd() / "view_cloud_3d.py",
                      Path.cwd() / "misc" / "view_cloud_3d.py",
                      Path.cwd().parent / "misc" / "view_cloud_3d.py"):
        if candidate.is_file():
            return candidate.resolve()
    return None


def _colour_dir():
    """Default folder for saved colour schemes: ``misc/conf/colours``.

    Anchored on view_cloud_3d.py (which _find_3d_script already locates) rather
    than on the working directory. A notebook may be run from misc/ OR from the
    repo root, and the repo root has its own conf/ -- the pipeline's, holding
    config.yaml -- which is not where these belong. Nothing is created here; the
    save dialog does that only if you actually save.
    """
    script = _find_3d_script()
    base = script.parent if script is not None else Path.cwd()
    return base / "conf" / "colours"


# Once a view is mouse-wheel zoomed, its limits are "fixed" while the axes keep
# aspect="equal", adjustable="datalim"; matplotlib then logs a WARNING on every
# draw ("Ignoring fixed x/y limits to fulfill fixed data aspect..."). That is the
# expected artifact of interactive equal-aspect zoom and carries no action for
# the user, so drop just that one message (other warnings pass through).
class _DropAspectLog(logging.Filter):
    def filter(self, record):
        return "to fulfill fixed data aspect" not in record.getMessage()


logging.getLogger("matplotlib.axes._base").addFilter(_DropAspectLog())


class CloudPanel:
    """One half of the window: load / inspect / filter / draw a single cloud."""

    def __init__(self, parent, title, colours=None, redraw_all=None):
        self.cloud = None           # the loaded cloud dict (see load_point_cloud)
        self.mask = None            # bool keep-mask from the filter (None = no filter)
        self._legend = None         # legend artist, replaced on every redraw
        self.selected_fields = None  # None = show every field in the stats box
        self.title = title          # "Cloud A" / "Cloud B", used in the 3-D title
        # The colour table. Normally the app hands both panels the SAME one, so a
        # value keeps its colour on both sides; a panel built on its own gets a
        # private one so it still works standalone.
        self.colours = colours if colours is not None else ColourScheme()
        # Called after a colour changes: the app redraws BOTH panels, since the
        # change affects whichever of them colours by that field.
        self.redraw_all = redraw_all if redraw_all is not None else self.redraw

        self.frame = ttk.LabelFrame(parent, text=title, padding=4)

        # -- row: load button + file name ----------------------------------
        row = ttk.Frame(self.frame)
        row.pack(fill="x")
        ttk.Button(row, text="Load\N{HORIZONTAL ELLIPSIS}", command=self.load_file).pack(side="left")
        self.file_label = ttk.Label(row, text="(no file loaded)", anchor="w")
        self.file_label.pack(side="left", fill="x", expand=True, padx=6)

        # -- info box -------------------------------------------------------
        self.info = ScrolledText(self.frame, height=9, width=40, font=("Consolas", 8),
                                 state="disabled", wrap="none")
        self.info.pack(fill="x", pady=(4, 4))
        # Stats-field picker, the colour editor and the 3-D view controls (kept
        # off the already busy colour-by row below).
        info_tools = ttk.Frame(self.frame)
        info_tools.pack(fill="x", pady=(0, 2))
        ttk.Button(info_tools, text="Fields shown\N{HORIZONTAL ELLIPSIS}",
                   command=self.open_field_chooser).pack(side="left")
        ttk.Button(info_tools, text="Colours\N{HORIZONTAL ELLIPSIS}",
                   command=self.open_colour_editor).pack(side="left", padx=(4, 0))
        ttk.Button(info_tools, text="3D view",
                   command=self.open_3d_view).pack(side="left", padx=(10, 2))
        ttk.Label(info_tools, text="3D max pts:").pack(side="left")
        self.max_points_3d_var = tk.StringVar(value=str(_DEFAULT_3D_MAX_POINTS))
        ttk.Entry(info_tools, textvariable=self.max_points_3d_var,
                  width=9).pack(side="left", padx=(2, 0))

        # -- row: colour-by field, max points, redraw + the pop-up buttons --
        row = ttk.Frame(self.frame)
        row.pack(fill="x", pady=(0, 2))
        ttk.Label(row, text="Colour by:").pack(side="left")
        self.color_var = tk.StringVar()
        self.color_combo = ttk.Combobox(row, textvariable=self.color_var,
                                        state="readonly", width=16)
        self.color_combo.pack(side="left", padx=(2, 10))
        self.color_combo.bind("<<ComboboxSelected>>", lambda _e: self.redraw())
        ttk.Label(row, text="Max points:").pack(side="left")
        self.max_points_var = tk.StringVar(value=str(_DEFAULT_MAX_POINTS))
        ttk.Entry(row, textvariable=self.max_points_var, width=9).pack(side="left", padx=(2, 10))
        ttk.Button(row, text="Redraw", command=self.redraw).pack(side="left")
        # Histogram / Unique read whatever the "Colour by" selector holds.
        ttk.Button(row, text="Histogram", command=self.open_histogram).pack(side="left", padx=(10, 2))
        ttk.Button(row, text="Unique\N{HORIZONTAL ELLIPSIS}", command=self.open_unique_values).pack(side="left")

        # -- rows: the filter -----------------------------------------------
        row = ttk.Frame(self.frame)
        row.pack(fill="x", pady=(0, 2))
        ttk.Label(row, text="Filter:").pack(side="left")
        self.filter_var = tk.StringVar()
        self.filter_combo = ttk.Combobox(row, textvariable=self.filter_var,
                                         state="readonly", width=16)
        self.filter_combo.pack(side="left", padx=(2, 10))
        self.mode_var = tk.StringVar(value="range")
        ttk.Radiobutton(row, text="Range", variable=self.mode_var,
                        value="range").pack(side="left")
        ttk.Radiobutton(row, text="Values", variable=self.mode_var,
                        value="values").pack(side="left")

        row = ttk.Frame(self.frame)
        row.pack(fill="x", pady=(0, 2))
        ttk.Label(row, text="min").pack(side="left")
        self.min_var = tk.StringVar()
        ttk.Entry(row, textvariable=self.min_var, width=9).pack(side="left", padx=(2, 6))
        ttk.Label(row, text="max").pack(side="left")
        self.max_var = tk.StringVar()
        ttk.Entry(row, textvariable=self.max_var, width=9).pack(side="left", padx=(2, 6))
        ttk.Label(row, text="values").pack(side="left")
        self.values_var = tk.StringVar()
        ttk.Entry(row, textvariable=self.values_var, width=14).pack(side="left", padx=(2, 6))
        ttk.Button(row, text="Apply", command=self.apply_filter).pack(side="left", padx=(4, 2))
        ttk.Button(row, text="Reset", command=self.reset_filter).pack(side="left")

        self.status = ttk.Label(self.frame, text="", anchor="w", foreground="#555")
        self.status.pack(fill="x", pady=(0, 2))

        # -- the figure: 2 x 3 grid = 5 views + legend/colourbar slot -------
        self.fig = Figure(figsize=(7.2, 6.2), dpi=100)
        self.axes = [self.fig.add_subplot(2, 3, i + 1) for i in range(5)]
        self.ax_extra = self.fig.add_subplot(2, 3, 6)
        self.ax_extra.axis("off")
        # The colourbar axes is created ONCE and only toggled visible/hidden:
        # destroying and recreating it on every redraw trips matplotlib's
        # Colorbar.remove() (it assumes gridspec-managed axes, ours is an inset).
        self.cax = self.ax_extra.inset_axes([0.35, 0.08, 0.14, 0.84])
        self.cax.set_visible(False)
        # Host the canvas in a frame with propagation disabled so the heavy
        # matplotlib widget can SHRINK. Otherwise, after the first draw(), the
        # canvas requests its full figure pixel size (~720x620) as a minimum and
        # tk refuses to shrink the window below it -> "resizing stops working
        # after loading a cloud". pack_propagate(False) decouples the frame's
        # size from the canvas request; fill/expand still lets it grow.
        plot_frame = ttk.Frame(self.frame, width=320, height=280)
        plot_frame.pack(fill="both", expand=True)
        plot_frame.pack_propagate(False)
        self.canvas = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)
        # Mouse wheel = zoom the subplot under the cursor (zoom-to-cursor).
        self.canvas.mpl_connect("scroll_event", self._on_scroll)
        # The standard matplotlib toolbar: zoom box, pan, back/forward, save PNG.
        toolbar_frame = ttk.Frame(self.frame)
        toolbar_frame.pack(fill="x")
        NavigationToolbar2Tk(self.canvas, toolbar_frame)

    # -- loading ------------------------------------------------------------

    def load_file(self):
        """Ask for a file, then load it (see load_path)."""
        path = filedialog.askopenfilename(title="Select a point cloud",
                                          filetypes=_FILE_TYPES)
        if not path:  # dialog cancelled
            return
        self.load_path(path)

    def load_path(self, path):
        """Load a cloud from a path -- the dialog-free half of load_file.

        Split out so a test (or a notebook cell) can populate a panel without
        clicking through a file dialog.
        """
        try:
            cloud = load_point_cloud(path)
        except Exception as exc:  # bad file, unsupported layout, ...
            messagebox.showerror("Load failed", f"{path}\n\n{exc}")
            return
        self.cloud = cloud
        self.mask = None
        self.selected_fields = None  # a new cloud has different fields -> show all
        self.file_label.config(text=cloud["name"])
        self._set_info(cloud_info_text(cloud, self.selected_fields))

        # Populate both field selectors; default to the first label-ish field.
        names = list(cloud["fields"])
        self.color_combo["values"] = names
        self.filter_combo["values"] = names
        default = next((f for f in _PREFERRED_FIELDS if f in cloud["fields"]),
                       names[0] if names else "")
        self.color_var.set(default)
        self.filter_var.set(default)
        self.redraw()

    def _set_info(self, text):
        self.info.config(state="normal")
        self.info.delete("1.0", "end")
        self.info.insert("1.0", text)
        self.info.config(state="disabled")

    # -- filtering ----------------------------------------------------------

    def apply_filter(self):
        if self.cloud is None:
            return
        field = self.filter_var.get()
        if field not in self.cloud["fields"]:
            messagebox.showwarning("Filter", f"Unknown field: {field!r}")
            return
        mode = self.mode_var.get()
        try:
            if mode == "range":
                vmin = float(self.min_var.get()) if self.min_var.get().strip() else None
                vmax = float(self.max_var.get()) if self.max_var.get().strip() else None
                self.mask = make_mask(self.cloud["fields"], field, "range",
                                      vmin=vmin, vmax=vmax)
            else:
                values = parse_values(self.values_var.get())
                if not values:
                    messagebox.showwarning("Filter", "Enter comma-separated values, e.g. 2, 3")
                    return
                self.mask = make_mask(self.cloud["fields"], field, "values", values=values)
        except ValueError as exc:
            messagebox.showerror("Filter", str(exc))
            return
        self.redraw()

    def reset_filter(self):
        self.mask = None
        self.redraw()

    def _field_values(self, field, respect_filter):
        """The field's values, optionally restricted to the active filter mask.

        Shared by the Histogram and Unique-values pop-ups so their "respect
        active filter" toggle behaves identically in one place.
        """
        col = self.cloud["fields"][field]
        if respect_filter and self.mask is not None:
            col = col[self.mask]
        return col

    def _selected_indices(self, max_pts):
        """Indices of the currently filtered points, downsampled to max_pts.

        Shared by the 2-D redraw and the 3-D export, which use DIFFERENT budgets
        but must agree on which points are "selected". The fixed seed means the
        3-D view is a superset of the 2-D subsample rather than an unrelated
        random draw.

        Returns (idx, n_filtered) -- n_filtered is the count BEFORE downsampling.
        """
        total = self.cloud["num_points"]
        keep = self.mask if self.mask is not None else np.ones(total, dtype=bool)
        idx = np.flatnonzero(keep)
        if idx.size > max_pts:
            rng = np.random.default_rng(_DOWNSAMPLE_SEED)
            idx = rng.choice(idx, size=max_pts, replace=False)
        return idx, int(keep.sum())

    # -- diagnostics pop-ups ------------------------------------------------

    def open_histogram(self):
        """Pop up a histogram of the current 'Colour by' field."""
        if self.cloud is None:
            return
        field = self.color_var.get()
        col = self.cloud["fields"].get(field)
        if col is None:
            messagebox.showwarning("Histogram", f"Unknown field: {field!r}")
            return
        if not np.issubdtype(col.dtype, np.number):
            messagebox.showinfo("Histogram", f"Field {field!r} is not numeric.")
            return

        win = tk.Toplevel(self.frame)
        win.title(f"Histogram \N{EM DASH} {field}")
        win.geometry("660x520")

        respect_var = tk.BooleanVar(value=False)   # default = whole cloud
        bins_var = tk.StringVar(value="50")
        controls = ttk.Frame(win, padding=4)       # top strip (populated below)
        controls.pack(fill="x")

        fig = Figure(figsize=(6.2, 4.4), dpi=100)
        ax = fig.add_subplot(1, 1, 1)
        canvas = FigureCanvasTkAgg(fig, master=win)
        canvas.get_tk_widget().pack(fill="both", expand=True)
        NavigationToolbar2Tk(canvas, win)

        def render():
            values = np.asarray(self._field_values(field, respect_var.get()))
            ax.clear()
            if values.size == 0:
                ax.set_title("no points in current selection")
                canvas.draw_idle()
                return
            uniq = np.unique(values) if np.issubdtype(values.dtype, np.integer) else None
            if uniq is not None and uniq.size <= _HIST_MAX_BARS:
                # One bar per integer value, centred on the value.
                edges = np.append(uniq, uniq[-1] + 1).astype(np.float64) - 0.5
                ax.hist(values, bins=edges, rwidth=0.9)
                ax.set_xticks(uniq)
            else:
                try:
                    n_bins = max(1, int(bins_var.get()))
                except ValueError:
                    n_bins = 50
                    bins_var.set("50")
                ax.hist(values, bins=n_bins)
            scope = "filtered" if (respect_var.get() and self.mask is not None) else "full cloud"
            ax.set_title(f"{field} \N{EM DASH} {values.size:,} pts ({scope})", fontsize=10)
            ax.set_xlabel(field)
            ax.set_ylabel("count")
            fig.tight_layout()
            canvas.draw_idle()

        ttk.Checkbutton(controls, text="respect active filter",
                        variable=respect_var, command=render).pack(side="left")
        ttk.Label(controls, text="bins:").pack(side="left", padx=(10, 2))
        ttk.Entry(controls, textvariable=bins_var, width=6).pack(side="left")
        ttk.Button(controls, text="Redraw", command=render).pack(side="left", padx=(6, 0))
        render()

    def open_unique_values(self):
        """Pop up a sortable table of the current 'Colour by' field's values."""
        if self.cloud is None:
            return
        field = self.color_var.get()
        if field not in self.cloud["fields"]:
            messagebox.showwarning("Unique values", f"Unknown field: {field!r}")
            return

        win = tk.Toplevel(self.frame)
        win.title(f"Unique values \N{EM DASH} {field}")
        win.geometry("440x580")

        respect_var = tk.BooleanVar(value=False)   # default = whole cloud
        controls = ttk.Frame(win, padding=4)
        controls.pack(fill="x")
        summary = ttk.Label(controls, text="", anchor="w")
        summary.pack(side="right", fill="x", expand=True, padx=(10, 0))

        cols = ("value", "count", "percent")
        tree = ttk.Treeview(win, columns=cols, show="headings")
        for c, txt, w in (("value", "Value", 150), ("count", "Count", 130),
                          ("percent", "% of points", 130)):
            tree.heading(c, text=txt, command=lambda cc=c: sort_by(cc))
            tree.column(c, width=w, anchor="e")
        vsb = ttk.Scrollbar(win, orient="vertical", command=tree.yview)
        tree.configure(yscrollcommand=vsb.set)
        vsb.pack(side="right", fill="y")
        tree.pack(fill="both", expand=True)

        state = {"rows": [], "sort": ("value", False)}  # rows = (value, count, pct)

        def repopulate():
            key, reverse = state["sort"]
            idx = {"value": 0, "count": 1, "percent": 2}[key]
            rows = sorted(state["rows"], key=lambda r: r[idx], reverse=reverse)
            tree.delete(*tree.get_children())
            for v, c, p in rows:
                vtxt = f"{v}" if isinstance(v, int) else f"{v:g}"
                tree.insert("", "end", values=(vtxt, f"{c:,}", f"{p:.3f}%"))

        def sort_by(col):
            key, reverse = state["sort"]
            state["sort"] = (col, not reverse if key == col else False)
            repopulate()

        def rebuild():
            values = np.asarray(self._field_values(field, respect_var.get()))
            tree.delete(*tree.get_children())
            if values.size == 0:
                summary.config(text="no points in selection")
                state["rows"] = []
                return
            uniq, counts = np.unique(values, return_counts=True)
            summary.config(text=f"{uniq.size:,} unique / {values.size:,} pts")
            if uniq.size > _UNIQUE_ROW_CAP:
                state["rows"] = []
                messagebox.showinfo(
                    "Unique values",
                    f"{field!r} has {uniq.size:,} unique values (> {_UNIQUE_ROW_CAP:,}).\n"
                    "Narrow it with a Range filter first, then reopen this table.")
                return
            total = values.size
            is_int = np.issubdtype(values.dtype, np.integer)
            state["rows"] = [((int(v) if is_int else float(v)), int(c), 100.0 * c / total)
                             for v, c in zip(uniq, counts)]
            repopulate()

        ttk.Checkbutton(controls, text="respect active filter",
                        variable=respect_var, command=rebuild).pack(side="left")
        rebuild()

    def open_field_chooser(self):
        """Pop up checkboxes choosing which fields the stats box lists."""
        if self.cloud is None:
            return
        names = list(self.cloud["fields"])
        current = set(names if self.selected_fields is None else self.selected_fields)

        win = tk.Toplevel(self.frame)
        win.title("Fields shown in statistics")
        win.geometry("260x480")

        ttk.Label(win, text="Show these fields:", padding=4).pack(fill="x")
        box = ttk.Frame(win, padding=(6, 0))
        box.pack(fill="both", expand=True)
        vars_by_name = {}
        for name in names:
            v = tk.BooleanVar(value=name in current)
            vars_by_name[name] = v
            ttk.Checkbutton(box, text=name, variable=v).pack(anchor="w")

        def set_all(flag):
            for v in vars_by_name.values():
                v.set(flag)

        def apply_and_close():
            chosen = [n for n in names if vars_by_name[n].get()]
            self.selected_fields = chosen or None  # nothing ticked -> show all
            self._set_info(cloud_info_text(self.cloud, self.selected_fields))
            win.destroy()

        btns = ttk.Frame(win, padding=4)
        btns.pack(fill="x")
        ttk.Button(btns, text="All", command=lambda: set_all(True)).pack(side="left")
        ttk.Button(btns, text="None", command=lambda: set_all(False)).pack(side="left", padx=(4, 0))
        ttk.Button(btns, text="Apply", command=apply_and_close).pack(side="right")
        ttk.Button(btns, text="Cancel", command=win.destroy).pack(side="right", padx=(0, 4))

    # -- the colour editor --------------------------------------------------

    def open_colour_editor(self):
        """Pop up per-value colour controls for the current 'Colour by' field.

        Lists exactly the values the legend shows -- the ones in the current
        selection -- because those are what you are looking at. Each row offers a
        hex box and the OS colour picker; the palette dropdown refills them all at
        once. Colours live in the shared ColourScheme, so every change redraws
        BOTH panels and also reaches the 3-D view.
        """
        if self.cloud is None:
            return
        field = self.color_var.get()
        col = self.cloud["fields"].get(field)
        if col is None:
            messagebox.showwarning("Colours", f"Unknown field: {field!r}")
            return

        # Colour the values the legend describes: the current selection.
        idx, _ = self._selected_indices(self._max_points())
        cvals = col[idx]
        if cvals.size == 0:
            messagebox.showinfo("Colours", "No points in the current selection.")
            return
        spec = color_spec(cvals, _MAX_LEGEND_CLASSES, scheme=self.colours, field=field)
        if not spec["discrete"]:
            # Same guidance as the Unique... table: narrow it down, then come back.
            messagebox.showinfo(
                "Colours",
                f"{field!r} is drawn as a continuous viridis ramp, so it has no "
                "per-value colours to set.\n\n"
                f"dtype {col.dtype}, {np.unique(cvals).size:,} distinct value(s) in the "
                "current selection.\n\n"
                f"Per-value colours need an INTEGER field with at most "
                f"{_MAX_LEGEND_CLASSES} distinct values in view. Narrow it with "
                "Filter \N{RIGHTWARDS ARROW} Values (or Range) and reopen this window.")
            return

        uniq, counts = np.unique(cvals, return_counts=True)

        win = tk.Toplevel(self.frame)
        win.title(f"Colours \N{EM DASH} {field}  ({self.title})")
        win.geometry("470x600")

        # -- top strip: the palette -----------------------------------------
        top = ttk.Frame(win, padding=4)
        top.pack(fill="x")
        ttk.Label(top, text="Palette:").pack(side="left")
        palette_var = tk.StringVar(value=self.colours.palette_of(field))
        ttk.Combobox(top, textvariable=palette_var, state="readonly",
                     values=list(PALETTES), width=17).pack(side="left", padx=(2, 6))

        # Bottom bars are packed BEFORE the scrolling body so that a window too
        # small for 20 rows clips the rows, never the buttons.
        status = ttk.Label(win, text="", anchor="w", foreground="#555")
        status.pack(side="bottom", fill="x", padx=6, pady=(0, 4))
        btns = ttk.Frame(win, padding=4)
        btns.pack(side="bottom", fill="x")

        body = ttk.Frame(win, padding=(6, 2))
        body.pack(fill="both", expand=True)
        for column, heading in ((0, "value"), (1, ""), (2, "hex"), (3, ""), (4, "points")):
            ttk.Label(body, text=heading, foreground="#555").grid(
                row=0, column=column, sticky="w", padx=2)

        rows = {}   # value -> {"swatch", "var", "entry"}

        def refresh_row(value):
            """Put the scheme's current colour into one row's swatch + hex box."""
            hexed = self.colours.colours_for(field, [value])[value]
            rows[value]["swatch"].config(background=hexed)
            rows[value]["var"].set(hexed)
            rows[value]["entry"].config(foreground="black")

        def apply_hex(value):
            """Validate + store what was typed. Silent no-op when unchanged."""
            typed = rows[value]["var"].get()
            hexed = parse_colour(typed)
            if hexed is None:
                rows[value]["entry"].config(foreground=_BAD_INPUT_FG)
                status.config(text=f"not a colour: {typed!r} \N{EM DASH} try #E9E56B, "
                                   "#abc, e9e56b or a name like 'red'")
                return
            if hexed == self.colours.known(field).get(value):
                rows[value]["entry"].config(foreground="black")
                return                      # nothing changed; do not redraw
            self.colours.set_colour(field, value, hexed)
            refresh_row(value)
            status.config(text=f"{field} = {value} \N{RIGHTWARDS ARROW} {hexed}")
            self.redraw_all()

        def pick(value):
            """The OS colour picker, seeded with the value's current colour."""
            current = self.colours.colours_for(field, [value])[value]
            _rgb, chosen = colorchooser.askcolor(color=current, parent=win,
                                                 title=f"{field} = {value}")
            if not chosen:
                return                      # dialog cancelled
            self.colours.set_colour(field, value, chosen)
            refresh_row(value)
            status.config(text=f"{field} = {value} \N{RIGHTWARDS ARROW} "
                               f"{self.colours.known(field)[value]}")
            self.redraw_all()

        for grid_row, (value, count) in enumerate(zip(uniq, counts), start=1):
            v = int(value)
            ttk.Label(body, text=str(v)).grid(row=grid_row, column=0, sticky="w", padx=2)
            # tk.Label, not ttk: only the classic widget takes a background colour.
            swatch = tk.Label(body, width=3, relief="solid", borderwidth=1)
            swatch.grid(row=grid_row, column=1, padx=2, pady=1)
            var = tk.StringVar()
            entry = ttk.Entry(body, textvariable=var, width=11, font=("Consolas", 9))
            entry.grid(row=grid_row, column=2, padx=2)
            entry.bind("<Return>", lambda _e, vv=v: apply_hex(vv))
            entry.bind("<FocusOut>", lambda _e, vv=v: apply_hex(vv))
            ttk.Button(body, text="Pick\N{HORIZONTAL ELLIPSIS}", width=6,
                       command=lambda vv=v: pick(vv)).grid(row=grid_row, column=3, padx=2)
            ttk.Label(body, text=f"{int(count):,}", foreground="#555").grid(
                row=grid_row, column=4, sticky="e", padx=(6, 2))
            rows[v] = {"swatch": swatch, "var": var, "entry": entry}
            refresh_row(v)

        def apply_palette():
            try:
                self.colours.apply_palette(field, palette_var.get(), uniq)
            except ValueError as exc:
                status.config(text=str(exc))
                return
            for v in rows:
                refresh_row(v)
            status.config(text=f"palette {palette_var.get()!r} applied to "
                               f"{len(self.colours.known(field))} value(s) of {field}")
            self.redraw_all()

        ttk.Button(top, text="Apply palette", command=apply_palette).pack(side="left")

        def save():
            target = _colour_dir() / f"{field}.json"
            path = filedialog.asksaveasfilename(
                parent=win, title="Save colour scheme", defaultextension=".json",
                initialfile=target.name, initialdir=str(target.parent),
                filetypes=_JSON_FILE_TYPES)
            if not path:
                return
            try:
                Path(path).parent.mkdir(parents=True, exist_ok=True)
                Path(path).write_text(
                    json.dumps(self.colours.to_json(field), indent=2) + "\n",
                    encoding="utf-8")
            except OSError as exc:
                messagebox.showerror("Colours", f"Could not save:\n{exc}")
                return
            status.config(text=f"saved {len(self.colours.known(field))} colour(s) "
                               f"to {Path(path).name}")

        def load():
            path = filedialog.askopenfilename(
                parent=win, title="Load colour scheme",
                initialdir=str(_colour_dir()), filetypes=_JSON_FILE_TYPES)
            if not path:
                return
            try:
                loaded_field, n_loaded, problems = self.colours.load_json(
                    json.loads(Path(path).read_text(encoding="utf-8")))
            except (OSError, ValueError, json.JSONDecodeError) as exc:
                messagebox.showerror("Colours", f"Could not load:\n{exc}")
                return
            for v in rows:
                refresh_row(v)
            palette_var.set(self.colours.palette_of(field))
            message = f"loaded {n_loaded} colour(s) for {loaded_field!r}"
            if loaded_field != field:
                message += (f" \N{EM DASH} this panel colours by {field!r}, so nothing "
                            "changed on screen")
            status.config(text=message)
            if problems:
                messagebox.showwarning(
                    "Colours", "The file had problems:\n\n" + "\n".join(problems[:10]))
            self.redraw_all()

        ttk.Button(btns, text="Load\N{HORIZONTAL ELLIPSIS}", command=load).pack(side="left")
        ttk.Button(btns, text="Save\N{HORIZONTAL ELLIPSIS}", command=save).pack(side="left", padx=(4, 0))
        ttk.Button(btns, text="Close", command=win.destroy).pack(side="right")
        status.config(text=f"{len(rows)} value(s) in view \N{EM DASH} colours are shared "
                           "by both panels and pinned per value")
        return win

    # -- the 3-D view -------------------------------------------------------

    def _max_points_3d(self) -> int:
        try:
            return max(1, int(self.max_points_3d_var.get()))
        except ValueError:
            self.max_points_3d_var.set(str(_DEFAULT_3D_MAX_POINTS))
            return _DEFAULT_3D_MAX_POINTS

    def build_3d_payload(self, idx):
        """(arrays, meta) describing the selection for the 3-D viewer process.

        Split out from open_3d_view so the export can be tested without a GUI.
        Colours come from the SAME color_spec() and the SAME ColourScheme the 2-D
        scatter uses, so a class keeps its colour when you switch to 3-D -- and
        keeps whatever you set in the Colours... editor.
        """
        fields = self.cloud["fields"]
        xyz = np.column_stack([fields["x"][idx], fields["y"][idx],
                               fields["z"][idx]]).astype(np.float32)

        color_field = self.color_var.get()
        cvals = fields.get(color_field)
        cvals = cvals[idx] if cvals is not None else fields["z"][idx]
        spec = color_spec(cvals, _MAX_LEGEND_CLASSES,
                          scheme=self.colours, field=color_field)

        meta = {
            "title": f"{self.title} \N{EM DASH} {self.cloud['name']}",
            "field": color_field or "z",
            "point_size": _POINT_SIZE_3D,
            "discrete": bool(spec["discrete"]),
        }
        arrays = {"xyz": xyz}
        if spec["discrete"]:
            arrays["rgb"] = spec["rgb"]
            meta["uniq"] = [int(v) for v in spec["uniq"]]
            meta["palette"] = [[int(round(c * 255)) for c in rgba[:3]]
                               for rgba in spec["palette"]]
        else:
            arrays["scalars"] = spec["scalars"].astype(np.float32)
        return arrays, meta

    def open_3d_view(self):
        """Open the current selection in a separate orbitable 3-D window.

        Runs ``misc/view_cloud_3d.py`` as its own process: VTK and tkinter each
        want the event loop, so nesting them would freeze this window. As a
        separate process the 2-D viewer stays live and both panels can have a
        3-D window open at once.
        """
        if self.cloud is None:
            return
        missing = [c for c in ("x", "y", "z") if c not in self.cloud["fields"]]
        if missing:
            messagebox.showwarning("3D view",
                                   f"Cannot show 3-D: missing coordinate field(s) {missing}")
            return

        script = _find_3d_script()
        if script is None:
            messagebox.showerror(
                "3D view",
                "Could not find view_cloud_3d.py.\n\nIt should sit next to this "
                "notebook in misc/. Current working directory:\n"
                f"{Path.cwd()}")
            return
        if importlib.util.find_spec("pyvista") is None:
            messagebox.showerror(
                "3D view",
                "The 3-D view needs PyVista + VTK, which are not installed.\n\n"
                "Install them with:\n"
                'cmd /c "mamba run -n aifor python -m pip install --user pyvista vtk"')
            return

        idx, n_filtered = self._selected_indices(self._max_points_3d())
        if idx.size == 0:
            messagebox.showinfo("3D view", "No points in the current selection.")
            return

        arrays, meta = self.build_3d_payload(idx)

        # Hand the points over through a temp .npz; the child deletes it once read.
        # delete=False: we close the handle here and the CHILD process owns the file.
        with tempfile.NamedTemporaryFile(suffix=".npz", prefix="cloud3d_",
                                         delete=False) as fh:
            payload = Path(fh.name)
        np.savez(payload, meta=json.dumps(meta), **arrays)

        # Keep the child's output: without this a failure (bad GPU driver, VTK
        # problem) would vanish and the window would just never appear.
        log_path = payload.with_suffix(".log")
        try:
            log = open(log_path, "w", encoding="utf-8")
            subprocess.Popen(
                [sys.executable, str(script), str(payload)],
                stdout=log, stderr=log,
                # Windows-only flag; getattr keeps this portable elsewhere.
                creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
            )
        except OSError as exc:
            messagebox.showerror("3D view", f"Could not start the 3-D viewer:\n{exc}")
            return

        self.status.config(
            text=f"3D view opened \N{EM DASH} {idx.size:,} of {n_filtered:,} "
                 f"filtered points   (log: {log_path.name})")

    # -- drawing ------------------------------------------------------------

    def _on_scroll(self, event):
        """Mouse-wheel zoom on the view under the cursor (zoom-to-cursor).

        Both axes scale by the SAME factor around the cursor position, so the
        point under the pointer stays fixed and the equal-aspect ratio of the
        view is preserved. The toolbar's Home button restores the base view.
        """
        ax = event.inaxes
        if ax not in self.axes or event.xdata is None or event.ydata is None:
            return  # not over one of the five spatial views
        f = (1.0 / _ZOOM_STEP) if event.button == "up" else _ZOOM_STEP
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        cx, cy = event.xdata, event.ydata
        ax.set_xlim(cx - (cx - x0) * f, cx + (x1 - cx) * f)
        ax.set_ylim(cy - (cy - y0) * f, cy + (y1 - cy) * f)
        self.canvas.draw_idle()

    def _max_points(self) -> int:
        try:
            return max(1, int(self.max_points_var.get()))
        except ValueError:
            self.max_points_var.set(str(_DEFAULT_MAX_POINTS))
            return _DEFAULT_MAX_POINTS

    def redraw(self):
        if self.cloud is None:
            return
        fields = self.cloud["fields"]
        missing = [c for c in ("x", "y", "z") if c not in fields]
        if missing:
            self.status.config(text=f"cannot draw: missing coordinate field(s) {missing}")
            return

        total = self.cloud["num_points"]
        # Filter + random downsample (fixed seed) so the view stays responsive
        # on multi-million-point plots -- shared with the 3-D export.
        idx, n_filtered = self._selected_indices(self._max_points())

        x, y, z = fields["x"][idx], fields["y"][idx], fields["z"][idx]

        # -- colours ---------------------------------------------------------
        color_field = self.color_var.get()
        cvals = fields.get(color_field)
        cvals = cvals[idx] if cvals is not None else z  # sensible fallback
        spec = color_spec(cvals, _MAX_LEGEND_CLASSES,
                          scheme=self.colours, field=color_field)
        discrete = spec["discrete"]
        if discrete:
            uniq, palette = spec["uniq"], spec["palette"]
            scatter_kw = dict(c=spec["rgba"])
        else:
            scatter_kw = dict(c=spec["scalars"], cmap="viridis")

        # -- the five views ---------------------------------------------------
        last_sc = None
        for ax, (label, view) in zip(self.axes, VIEWS):
            ax.clear()
            u, v = project(x, y, z, view)
            last_sc = ax.scatter(u, v, s=1.5, linewidths=0, **scatter_kw)
            ax.set_title(label, fontsize=9)
            ax.tick_params(labelsize=7)
            ax.set_aspect("equal", adjustable="datalim")

        # -- legend / colourbar in the 6th slot -------------------------------
        if self._legend is not None:   # drop last redraw's legend
            self._legend.remove()
            self._legend = None
        self.cax.clear()
        if discrete:
            self.cax.set_visible(False)
            handles = [Line2D([], [], marker="o", linestyle="", markersize=6,
                              markerfacecolor=palette[i], markeredgecolor="none",
                              label=str(val))
                       for i, val in enumerate(uniq)]
            self._legend = self.ax_extra.legend(
                handles=handles, loc="center", fontsize=8,
                title=color_field, title_fontsize=9,
                ncols=1 + uniq.size // 11, frameon=False)
        elif last_sc is not None:
            self.cax.set_visible(True)
            self.fig.colorbar(last_sc, cax=self.cax)
            self.cax.set_title(color_field, fontsize=8)
            self.cax.tick_params(labelsize=7)

        self.fig.tight_layout()
        self.canvas.draw_idle()
        self.status.config(
            text=f"shown {idx.size:,} / filtered {n_filtered:,} / total {total:,}"
                 + ("" if self.mask is None else "   [filter active]")
        )


class PointCloudCompareApp:
    """The main window: two independent CloudPanels side by side."""

    def __init__(self, root):
        root.title("Split Point-Cloud Viewer \N{EM DASH} 2-cloud diagnostic")
        root.geometry("1760x1000")
        root.minsize(900, 600)   # sane floor; panels can shrink to here after loading
        container = ttk.Frame(root, padding=4)
        container.pack(fill="both", expand=True)
        # ONE colour scheme for the whole window: a value keeps its colour in both
        # panels, which is the point of a side-by-side comparison. Both panels get
        # redraw_all, so editing a colour updates whichever panel shows that field.
        self.colours = ColourScheme()
        self.left = CloudPanel(container, "Cloud A",
                               colours=self.colours, redraw_all=self.redraw_all)
        self.right = CloudPanel(container, "Cloud B",
                                colours=self.colours, redraw_all=self.redraw_all)
        self.left.frame.pack(side="left", fill="both", expand=True, padx=(0, 3))
        self.right.frame.pack(side="left", fill="both", expand=True, padx=(3, 0))

    def redraw_all(self):
        """Redraw every panel that has a cloud (after a shared colour changed)."""
        for panel in (getattr(self, "left", None), getattr(self, "right", None)):
            if panel is not None and panel.cloud is not None:
                panel.redraw()


def launch_viewer(auto_close_ms=None):
    """Open the viewer window. Blocks until the window is closed.

    auto_close_ms: close the window automatically after N milliseconds --
    only used by automated smoke tests, leave as None for normal use.
    """
    root = tk.Tk()
    app = PointCloudCompareApp(root)
    if auto_close_ms is not None:
        root.after(auto_close_ms, root.destroy)
    root.mainloop()
    return app


print("GUI defined")

GUI defined


---
## Launch the viewer

Running the next cell opens the desktop window. **The kernel stays busy until you
close the window** — that is normal for a tkinter GUI inside a notebook.

Tips:
- Load a different file at any time with **Load…** (each panel is independent).
- **Mouse-wheel** over any spatial view zooms it at the cursor; the **matplotlib
  toolbar** under each figure gives box-zoom / pan / Home / save-as-PNG.
- **3D view** opens the current selection in its own window: drag = rotate,
  scroll = zoom, middle-drag = pan, `r` = reset, `d` = depth shading, `q` = close.
  It is a separate process, so this window keeps working and both panels can have a
  3-D window open at once. Filter first, then open it — it shows what the filter
  selected, up to **3D max pts**.
- **Colours…** sets one colour per value of the current **Colour by** field. The usual
  move on our data: colour by `semantic_seg` → **Colours…** → palette **ForAINet
  classes** → *Apply palette* → **Save…**, and every later session can **Load…** the
  same file. Type a hex (`#E9E56B`, `#abc`, or a name like `red`) or use **Pick…**.
  Both panels and the 3-D view follow immediately.
- Colours are **pinned per value and shared by both panels**, so a class keeps its
  colour when you filter, and Cloud A and Cloud B agree even if one of them is missing
  a class. To recolour a single tree, filter `tree_ID` down to a few ids first — then
  those ids appear as rows in **Colours…**.
- **Histogram** and **Unique…** open a pop-up for the current **Colour by** field.
  Tick *respect active filter* in the pop-up to restrict it to the filtered points.
- **Unique…** is the quick way to find real ids in a wide field: colour by `tree_ID`,
  open it, read off an existing id, then **Filter → Values** that id — and then
  **3D view** to see that single tree from any angle.
- **Fields shown…** trims the statistics box — untick the constant/noise fields.
- **Filter → Values** with e.g. `2, 3` on a `Class` field shows only those labels;
  **Range** with min/max works better for continuous fields (`z`, `dist_axes`).
- Raise **Max points** for more detail (slower), lower it for speed.
- The window can be resized freely, including **after** a cloud is loaded.

In [20]:
# Opens the viewer window; the ker
# nel is busy until the window is closed.
app = launch_viewer()